# Class 02: Cow Behaviour Classification

*AI4AS — Pitfalls in Machine Learning (exercise notebook)*


## Introduction

In this notebook we predict cow behaviour from **RumiWatch** sensor data. The data contain movement, pressure, and temperature measurements from one dairy cow. This is important: the model can be tested on later days from the same cow, but this does not prove that it will work equally well on other cows.

The main learning point is that the behaviour classes are not equally common. This makes model evaluation harder than it first looks.

You will work with a time-based train-test split, a majority-class baseline, a Random Forest model, SMOTE, threshold tuning, and cost-sensitive learning. The aim is not only to get a good score, but also to understand which classes are easy or difficult to predict.

Please work through the notebook **in order**. The numbered headings show the structure. Light-blue boxes are tasks. Amber boxes explain how to read the output.

### Official documentation

When you are unsure about a function, start with the official documentation.

| Library | Where to look |
|---------|----------------|
| **scikit-learn** | [User guide](https://scikit-learn.org/stable/user_guide.html) and [API reference](https://scikit-learn.org/stable/api/index.html) |
| **pandas** | [User guide](https://pandas.pydata.org/docs/user_guide/index.html) |
| **matplotlib** | [Tutorials](https://matplotlib.org/stable/tutorials/index.html) |
| **seaborn** | [API reference](https://seaborn.pydata.org/api.html) |
| **imbalanced-learn** | [Documentation](https://imbalanced-learn.org/stable/) |

## Table of contents

1. **Setup**: load sensor data
2. **Data exploration**: `2.1` and `2.2`
3. **Train-test split**: `3.1`
4. **Feature engineering**: `4.1`
5. **Baseline classifier**: `5.1`
6. **Random Forest pipeline**: `6.1` and `6.2`
7. **Class imbalance (advanced)**: SMOTE `7.1`
8. **Threshold tuning (advanced)**: `8.1`
9. **Cost-sensitive learning (advanced)**: `9.1`
10. **Confusion matrix evaluation**: `10.1`
11. **Feature importance**: `11.1`
12. **Interpretation plot**: `12.1`


## 1. Setup

### 1.1 Import required libraries

Import necessary libraries for data processing, model training, and visualization.

In [1]:
import zipfile

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler

### 1.2 Load data

Below the compressed dataset is loaded into a pandas DataFrame. The dataset contains raw sensor measurements and behaviour labels from one dairy cow equipped with a RumiWatch sensor.

This matters for interpretation. Later we will split the last 5 days as test data. That is a good temporal split, because the model is tested on future measurements. However, it is still future data from the same cow. The notebook therefore tests generalisation over time within one animal, not generalisation to new animals.

In [2]:
zip_path = "../data/raw/rwu_275a_1_convert_10hz.zip"
csv_filename = "rwu_275a_1_convert_10hz.csv"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open(csv_filename) as f:
        df_data = pd.read_csv(f, sep=';')


# Drop columns where all values are NaN
df_data = df_data.dropna(axis=1, how="all")

print(df_data.head())


: 

### 1.3 Estimate the sampling frequency

Before we use time windows, it is useful to check the sampling frequency from the data itself. The file name says `10hz`, but we should not only rely on the file name. The `time` column contains measurements every 0.1 seconds, which corresponds to 10 measurements per second.

The code below estimates this frequency from the median time difference between consecutive rows. We use the median because it is robust to an occasional irregular time stamp.

In [ ]:
# Estimate the sampling frequency from the timestamp column.
time_values = pd.to_datetime(df_data["time"], format="%d.%m.%Y %H:%M:%S.%f")
time_step_seconds = time_values.diff().dt.total_seconds().median()
sampling_rate_hz = int(round(1 / time_step_seconds))

print(f"Estimated time step: {time_step_seconds:.3f} seconds")
print(f"Estimated sampling frequency: {sampling_rate_hz} Hz")

### 1.4 Drop habituation period

To avoid unreliable sensor data during habituation, the first 2 hours of measurements are dropped. We now use the estimated sampling frequency to calculate how many rows correspond to 2 hours.

In [ ]:
# Drop first 2 hours of data.
habituation_rows = 2 * 60 * 60 * sampling_rate_hz
df_data = df_data.iloc[habituation_rows:]

## 2. Data exploration

### Overview of the variables

Below an overview of the dataset is provided.

| Variable       | Description                                  |
|----------------|----------------------------------------------|
| time           | Logging date and time (DD.MM.YYYY, hh:mm:ss) |
| pressure       | Pressure value                               |
| move_x         | Acceleration in x-axis                       |
| move_y         | Acceleration in y-axis                       |
| move_z         | Acceleration in z-axis                       |
| temperature    | Temperature value                            |
| peak_time      | RumiWatch Unit internal counter              |
| classification | Classification of the metered data           |

**Classification mapping:**

The integer labels in the `classification` column correspond to the following cow behaviours:

| Label | Meaning        |
|-------|----------------|
| 0     | Other activity |
| 1     | Ruminating     |
| 2     | Feed intake    |
| 4     | Water intake   |

The goal is to predict the `classification` label using the sensor measurements as features. In this notebook, we treat this column as the **reference label**: it is the label against which our model predictions are compared.

It is important to distinguish this from direct observation. The notebook data do not prove, by themselves, whether these labels came from human annotation, video annotation, or a proprietary RumiWatch classification algorithm. For the modelling exercise, we use the labels as the available reference, but in a real study the origin and reliability of these labels should be checked carefully.

### 2.1 Summary of the dataset

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**2.1 — TASK**

Make a summary of the dataset.

</div>

In [ ]:
# Add your code here

# Hints:
# - Use `df_data.describe()` or similar pandas summary methods.
# - Check row counts, ranges of sensor columns, and missing values.
# - Focus on understanding scale and variability before modelling.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation — dataset summary**

**How to read this**
- A numeric summary shows the scale and spread of each sensor channel before windowing.
- Very large row counts are expected because the data are sampled many times per second.
- Check for missing values or implausible ranges before feature engineering.

**This notebook's output**
- Millions of rows per sensor column; pressure mean ≈ 1506, accelerometer axes centred near zero with substantial spread.
- Temperature is relatively stable compared to movement channels.

</div>

### 2.2 Class distributions

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**2.2 — TASK**

Visualize the class distributions.

</div>

In [ ]:
# Add your code here

# Hints:
# - Use `value_counts()` on `classification` and plot a bar chart.
# - Label axes and title clearly; use behaviour names if `class_mapping` is already defined, otherwise class integers.
# - Note which class is majority — this matters for the baseline later.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: class imbalance**

**How to read this**
- First check how often each class occurs.
- If one class is much larger than the others, accuracy can be misleading.
- Rare classes are usually harder for the model to learn.

**In this notebook**
- After dropping the first 2 hours, the raw data still contain millions of rows because the estimated sampling frequency is 10 Hz.
- The class counts are about 6.92M rows for class 0, 4.58M for class 1, 1.27M for class 2, and 73k for class 4.
- Class 0 is clearly the largest class. A simple model can therefore get a reasonable accuracy by predicting class 0 very often.
- Class 4 is very rare. In the next step it is merged with class 2.

</div>

### 2.3 Merge ingestive classes

For this exercise we will combine class 4 into 2 as "ingestive behaviour".

In [ ]:
# Add class 4 to "feed intake" (class 2)
df_data["classification"] = df_data["classification"].replace(4, 2)


### 2.4 Behaviour labels

To facilitate interpretation we will add a class mapping.

In [ ]:
# Map class numbers to behaviour labels
class_mapping = {
    0: "Other activity",
    1: "Rumination",
    2: "Ingestive behaviour"
}

# Add readable behaviour column
df_data["behaviour"] = df_data["classification"].map(class_mapping)

# Quick check
print(df_data[["classification", "behaviour"]].drop_duplicates().sort_values("classification"))

### 2.5 Visualize sensor data over time

Plot with 1 hour of movement data over time:
- Series: `move_x`, `move_y`, and `move_z` and `pressure`.
- Shaded vertical background regions to indicate classification labels.

In [ ]:
n_obs = 1 * 60 * 60 * sampling_rate_hz
time = pd.to_datetime(df_data["time"], format="%d.%m.%Y %H:%M:%S.%f")[:n_obs]
move_x = (df_data['move_x'][:n_obs] - df_data['move_x'][:n_obs].min()) / (df_data['move_x'][:n_obs].max() - df_data['move_x'][:n_obs].min())  # Center move_x around zero
move_y = (df_data['move_y'][:n_obs] - df_data['move_y'][:n_obs].min()) / (df_data['move_y'][:n_obs].max() - df_data['move_y'][:n_obs].min())  # Center move_y around zero
move_z = (df_data['move_z'][:n_obs] - df_data['move_z'][:n_obs].min()) / (df_data['move_z'][:n_obs].max() - df_data['move_z'][:n_obs].min())  # Center move_z around zero
pressure = (df_data['pressure'][:n_obs] - df_data['pressure'][:n_obs].min()) / (df_data['pressure'][:n_obs].max() - df_data['pressure'][:n_obs].min())  # Center pressure around zero
classes = df_data['classification'][:n_obs]
unique_classes = np.unique(classes)

fig, ax = plt.subplots(figsize=(18, 5))

colors = plt.cm.tab10.colors

# Plot classification background bands
for cls in unique_classes:
    mask = classes == cls
    y_min = pressure.min() - 0.1
    y_max = pressure.max() + 0.1

    ax.fill_between(
        time,
        y_min,
        y_max,
        where=mask.to_numpy(),
        color=colors[int(cls) % len(colors)],
        alpha=0.15,
        label=f'Class {cls}'
    )

# Plot move_x, move_y, move_z, and pressure on the same axis
ax.plot(time, move_x, label='move_x', color='tab:blue', linewidth=0.5)
ax.plot(time, move_y, label='move_y', color='tab:orange', linewidth=0.5)
ax.plot(time, move_z, label='move_z', color='tab:green', linewidth=0.5)
ax.plot(time, pressure, label='pressure', color='tab:red', linewidth=0.05)

# Combine legends (only one axis now)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_xlabel('Time')
ax.set_ylabel('Movement')

plt.tight_layout()
plt.show()

## 3. Train–test split

In this section, you will select the features and classification for model training, and perform a train/test split.

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**3.1 — TASK**

Split the dataset into a training set and a test set. Assign them to `train` and `test`.

Use the last 5 days of measurements as the test set. This respects the temporal order: the model is trained on earlier measurements and tested on later measurements.

</div>

In [ ]:
# Add your code here

# Hints:
# - Use the provided formula: last 5 days × 24 h × 3600 s × `sampling_rate_hz` rows for the test set.
# - Assign `train` and `test` DataFrames with the same columns as `df_data`.
# - Keep time order: do not shuffle before splitting.
# - Remember: this split tests future measurements from the same cow.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: temporal split**

**How to read this**
- With time-series data, do not randomly shuffle the rows before splitting.
- A time-based split is appropriate here: train on earlier measurements and test on later measurements.
- This avoids a common temporal leakage problem.
- The remaining limitation is not the time split. The limitation is that all measurements come from one cow.

**In this notebook**
- The training set has shape `(8521230, 5)`. The test set has shape `(4320000, 5)`.
- The test set is the last 5 days of measurements. The number of rows is calculated with the estimated `sampling_rate_hz`.
- This means the model is tested on future data from the same cow.
- To know whether the model works for other cows, we would need data from other animals and ideally a split by animal.

</div>

## 4. Feature engineering: Data Segmentation and Feature Calculation

In order to perform classification, a first step is segmentation of the raw sensor data into fixed time windows and calculation of statistical features from each window. This approach transforms continuous time-series data into a structured feature matrix suitable for machine learning models.

### Windowing and Feature Extraction

The sensor data will be segmented into 60-second windows. For each window, we extract statistical features from the accelerometer (move_x, move_y, move_z) and pressure measurements:

**Features extracted per window:**
For this exercise we only calculated a limited number of features, feel free to add features. The Python package tsfresh allows to calculate many more features in an automated way.

Some important **Derived features:**: 

Signal Vector Magnitude (SVM): $\sqrt{\bar{x}^2 + \bar{y}^2 + \bar{z}^2}$
Signal Magnitude Area (SMA): $\sum(|\bar{x}| + |\bar{y}| + |\bar{z}|)$

Overall Dynamic Body Acceleration (ODBA): 
A proxy for activity level, computed as the sum of absolute dynamic acceleration across all three axes:

$$\mathrm{ODBA} = |a_x - \mu_x| + |a_y - \mu_y| + |a_z - \mu_z|$$

where $a_x, a_y, a_z$ are instantaneous accelerations and $\mu_x, \mu_y, \mu_z$ are their smoothed/static components.

The most frequent classification label within each window is assigned as the window's class label. This generates a balanced feature matrix with the engineered features and corresponding behaviour labels.

In [ ]:
def compute_odba_xyz(ax, ay, az, alpha=0.10):
    """
    Compute ODBA from tri-axial accelerometer data and return
    both the ODBA time series and its mean.

    Parameters
    ----------
    ax, ay, az : array-like
        Acceleration measurements along x, y, z axes.
    alpha : float, default=0.10
        EMA smoothing factor.

    Returns
    -------
    odba : np.ndarray
        ODBA time series.
    mean_odba : float
        Mean ODBA over the entire recording.
    """

    ax = np.asarray(ax, dtype=float)
    ay = np.asarray(ay, dtype=float)
    az = np.asarray(az, dtype=float)

    if not (len(ax) == len(ay) == len(az)):
        raise ValueError("ax, ay, and az must have the same length")

    def ema(x):
        mu = np.empty_like(x)
        mu[0] = x[0]
        for i in range(1, len(x)):
            mu[i] = alpha * x[i] + (1 - alpha) * mu[i - 1]
        return mu

    sx = ema(ax)
    sy = ema(ay)
    sz = ema(az)

    dx = ax - sx
    dy = ay - sy
    dz = az - sz

    odba = np.abs(dx) + np.abs(dy) + np.abs(dz)
    mean_odba = float(np.mean(odba))

    return mean_odba

In [ ]:
def extract_window_features(
    df,
    n_seconds=60,
    sampling_rate=sampling_rate_hz,
    label_col="classification"
):
    window_size = n_seconds * sampling_rate
    n_windows = len(df) // window_size

    features = []
    labels = []
    purities = []

    for i in range(n_windows):
        window = df.iloc[i * window_size:(i + 1) * window_size]

        # =========================
        # PRESSURE FEATURES
        # =========================
        pressure_series = window["pressure"]
        mean_pressure = np.mean(window["pressure"])
        range_pressure = np.max(window["pressure"]) - np.min(window["pressure"])

        # =========================
        # ACCELEROMETER FEATURES
        # =========================
        acc_mag = np.sqrt(
            window["move_x"]**2 +
            window["move_y"]**2 +
            window["move_z"]**2
        )
        mean_odba = compute_odba_xyz(window["move_x"], window["move_y"], window["move_z"], alpha=0.15)
                
        svm = np.mean(acc_mag)
        svm_sum = np.sum(acc_mag)
        svm_std = np.std(acc_mag)

        sma = (
            np.abs(window["move_x"]).mean()
            + np.abs(window["move_y"]).mean()
            + np.abs(window["move_z"]).mean()
        )

        # =========================
        # FEATURE VECTOR
        # =========================
        features.append({
            # pressure stats
            "mean_pressure": mean_pressure,
            "range_pressure": range_pressure,

            # movement
            "odba": mean_odba,

            "svm": svm,
            "svm_sum": svm_sum,
            "svm_std": svm_std,
            "sma": sma
        })

        label_distribution = window[label_col].value_counts(normalize=True)
        majority_label = label_distribution.index[0]
        purity = label_distribution.iloc[0]

        labels.append(majority_label)
        purities.append(purity)

    return pd.DataFrame(features), pd.Series(labels), pd.Series(purities, name="purity")

We apply the feature extraction function to the training and test sets.

In [ ]:
X_train, y_train, w_train_purity = extract_window_features(train)
X_test, y_test, w_test_purity = extract_window_features(test)

print(f'Train shape {X_train.shape}, Test shape {X_test.shape}')
print('Train purity summary:')
display(w_train_purity.describe())

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**4.1 — TASK**

Visualize the features in relation to the target variable.

</div>

In [ ]:
# Add your code here

# Hints:
# - Loop over engineered feature columns (e.g. `odba`, `mean_pressure`, `svm`).
# - Use boxplots or violin plots grouped by `classification` / behaviour label.
# - Compare separability across classes — features that overlap heavily are harder to learn.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: features vs target**

**How to read this**
- Exploratory data analysis means looking at the data before fitting a model.
- Here we use simple plots to see whether the engineered features differ between behaviour classes.
- Boxplots by class reveal overlap. If the boxes overlap strongly, the classes will be harder to separate.
- Activity-related features such as ODBA and SVM often differ most between rumination and other states.

**In this notebook**
- Plots for `odba`, pressure statistics, and SVM-derived features show partial separation between classes.
- Some features overlap substantially. No single feature perfectly separates the classes.
- This motivates using a multivariate model, such as a Random Forest, instead of one feature alone.

</div>

## 5. Baseline model

Before training a real classifier, we make a very simple reference model. This is called a **baseline**. A baseline tells us what score we can get with almost no learning.

Here we use a **majority class baseline**. This model first looks at `y_train`, the behaviour labels in the training set. It finds the class that occurs most often. Then it predicts that same class for every row in the test set.

Important: this baseline does **not** use the sensor features in `X_train` or `X_test`. It ignores movement and pressure completely. That is why it is useful: any real model should perform better than this very simple rule.

### Reading classification metrics

Before we evaluate the baseline, it helps to know what the printed scores mean. In a multiclass problem, we can read the report **one class at a time**. For example, when we inspect `Rumination`, we temporarily ask: "Rumination or not rumination?"

For one class, the basic counts are:

| Term | Meaning for one class |
|------|------------------------|
| **True positive (TP)** | The window truly belongs to this class, and the model predicts this class. |
| **False positive (FP)** | The window does not belong to this class, but the model predicts this class. This is a false alarm. |
| **False negative (FN)** | The window belongs to this class, but the model predicts another class. This is a missed example. |
| **True negative (TN)** | The window does not belong to this class, and the model also predicts another class. |

**Accuracy** is the fraction of all test windows that are predicted correctly. Using the same TP, FP, FN, and TN notation:

```text
accuracy = (TP + TN) / (TP + TN + FP + FN)
```

Accuracy counts both correctly found examples of the class (TP) and correctly rejected examples outside the class (TN). It is easy to understand, but it can be misleading when one class is much larger than the others. A model can get a reasonable accuracy by mostly predicting `Other activity`.

**Precision** asks: when the model predicts this class, how often is it correct?

```text
precision = TP / (TP + FP)
```

High precision means few false positives. In this notebook, high precision for `Rumination` would mean: when the model says "rumination", it is usually correct.

**Recall** asks: of all true examples of this class, how many did the model find?

```text
recall = TP / (TP + FN)
```

High recall means few false negatives. High recall for `Rumination` would mean: most real rumination windows are found by the model.

**F1-score** combines precision and recall into one number:

```text
F1 = 2 * (precision * recall) / (precision + recall)
```

F1 is high only when both precision and recall are high. If either precision or recall is low, F1 becomes low. This is why F1 is useful for imbalanced classification.

In the classification report, each class gets its own F1-score. For example, class 0 has one F1-score, class 1 has one F1-score, and class 2 has one F1-score.

**Macro-F1** is the average of the class-level F1-scores:

```text
macro-F1 = (F1_class0 + F1_class1 + F1_class2) / 3
```

Macro-F1 gives each class the same weight. This is useful for imbalanced data, because the small class matters just as much as the large class.

**Weighted-F1** is different. It also averages the class-level F1-scores, but larger classes get more weight. That is why weighted-F1 is often closer to accuracy when one class is much larger than the others.

**Support** is the number of true examples of each class in the test set:

```text
support = TP + FN
```

In this notebook, support tells us how many test windows truly belong to `Other activity`, `Rumination`, and `Ingestive behaviour`.

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**5.1 — TASK**

Build the majority class baseline.

Use `y_train` to find the most common behaviour class in the training data. Then create predictions for the test set by predicting this same class for every test window.

Your prediction vector should have the same length as `y_test`. Store it in `y_pred_majority`. Then compare `y_pred_majority` with the true test labels `y_test` and report the accuracy.

You do not need `X_train` or `X_test` for this baseline, because the baseline ignores all sensor features.

</div>

In [ ]:
# Add your code here

# Hints:
# - `y_train` contains the true behaviour labels for the training windows.
# - Find the most common label in `y_train` with `value_counts().idxmax()` or `mode()`.
# - `y_test` contains the true behaviour labels for the test windows.
# - Create `y_pred_majority`: one prediction for each value in `y_test`, always using the majority class.
# - Compare `y_pred_majority` with `y_test` using `accuracy_score`.
# - You can also print `classification_report(y_test, y_pred_majority)` to see what happens per class.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: majority baseline**

**How to read this**
- A baseline is a simple reference point. It helps us judge whether a real model is useful.
- The majority baseline predicts only the most common training class.
- Because it ignores the sensor data, it cannot learn differences between behaviours.
- On imbalanced data, this baseline can still get a moderate accuracy. That is why accuracy alone can be misleading.

**In this notebook**
- The majority class is class 0: Other activity.
- The baseline predicts class 0 for all 7200 test windows.
- Accuracy is about 0.518, because many test windows are indeed class 0.
- Macro-F1 is only about 0.227. This is low because the model completely fails on the other classes.
- Rumination and ingestive behaviour have 0 precision and 0 recall, because the baseline never predicts them.
- The `support` column shows how many true test windows there are for each class.

</div>

## 6. Random Forest pipeline

For the next step, you will train a Random Forest classifier.

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**6.1 — TASK**

Build a classification pipeline that applies `StandardScaler` and then `RandomForestClassifier`.

Use a pipeline, fit on the training data and use it for prediction on the test data.

</div>

In [ ]:
# Add your code here

# Hints:
# - Define `rf = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestClassifier(...))])`
# - Use `random_state=42` and `n_jobs=-1` as in the solution for reproducibility
# - Fit on `X_train`, `y_train` only
# rf = Pipeline([


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: Random Forest pipeline**

**What is the object shown above?**
- The object shown above is a fitted scikit-learn `Pipeline`.
- Jupyter displays scikit-learn models as a visual summary when the model object is the last output of a cell.
- In this cell, the last line is `rf.fit(X_train, y_train)`. The `.fit()` method trains the pipeline and returns the fitted pipeline object, so Jupyter shows it.

**How is the pipeline created?**
- A pipeline is a sequence of named steps.
- Here the first step is `StandardScaler`, called `scaler`.
- The second step is `RandomForestClassifier`, called `rf`.
- When we call `rf.fit(X_train, y_train)`, scikit-learn first fits the scaler on `X_train`, transforms `X_train`, and then fits the Random Forest on the scaled features.

**How do we use it?**
- Use `rf.predict(X_test)` to predict behaviour classes for the test windows.
- The pipeline automatically applies the same scaling to `X_test` before making predictions.
- You can inspect individual steps with `rf.named_steps["scaler"]` and `rf.named_steps["rf"]`.
- Later, when we plot feature importances, we use `rf_smote.named_steps["rf"].feature_importances_` to access the Random Forest inside another pipeline.

**Why use a pipeline?**
- It keeps preprocessing and modelling together in one object.
- It reduces the risk of data leakage, because preprocessing is fitted only on the training data.
- It makes later prediction easier: one call to `.predict()` does all required steps in the right order.

**Why set `random_state=42`?**
- A Random Forest contains random choices. For example, each tree is trained on a bootstrap sample of the training data, and each split considers a random subset of features.
- Later, SMOTE also uses randomness when it creates synthetic examples for the minority classes.
- Setting `random_state=42` makes these random choices repeatable when the notebook is run again in the same software environment.
- Small numerical differences can still occur across scikit-learn versions or computer systems, especially when models run in parallel with `n_jobs=-1`. Therefore, focus on the pattern of the results, not on the last decimal.

</div>

<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: class labels and probabilities**

A Random Forest classifier can give two kinds of output.

1. **Class prediction** with `.predict(X_test)`. This returns one final class label for each test window.
2. **Class probabilities** with `.predict_proba(X_test)`. This returns one probability for each possible class.

Where do these probabilities come from? A Random Forest is made of many decision trees. Each tree gives a class prediction. The forest combines the trees. If many trees support class 0, the probability for class 0 becomes high. If only a few trees support class 2, the probability for class 2 becomes low. In simple terms, the probabilities reflect how strongly the trees vote for each class.

The performance metrics in the next task use the final class labels from:

```python
y_pred_rf = rf.predict(X_test)
```

For a multiclass Random Forest, `.predict()` usually chooses the class with the **highest predicted probability**. For example, if the probabilities for one test window are:

```text
class 0: 0.60
class 1: 0.30
class 2: 0.10
```

then `.predict()` returns class 0.

In this default prediction step, we do not manually choose a separate threshold for each class. The model simply takes the class with the largest probability. Later, in the ROC and precision-recall curves, we will use `.predict_proba()` to study what happens when different thresholds are tried.

</div>

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**6.2 — TASK**

Apply the random forest model (`rf`) on the test data and store the predictions in `y_pred_rf`. Evaluate the performance of the model and report the accuracy, and provide a classification report.

</div>

In [ ]:
# Add your code here

# Hints:
# - Call `rf.predict(X_test)` and store in `y_pred_rf`.
# - Use `accuracy_score` and `classification_report(y_test, y_pred_rf)`.
# - Compare to the majority baseline — all classes should improve.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: Random Forest evaluation**

**How to read this**
- Compare the Random Forest with the majority-class baseline.
- Accuracy tells you the overall score, but macro-F1 gives each class equal weight.
- Always inspect the class-level F1 scores. The smallest class is often the most difficult one.

**In this notebook**
- Accuracy is about 0.880 and macro-F1 is about 0.807.
- Classes 0 and 1 perform well, with F1 scores around 0.94 and 0.88.
- Class 2 is clearly harder, with an F1 score around 0.61.
- The model is much better than the baseline, but it still struggles with ingestive behaviour.

</div>

## 7. Class imbalance (advanced)

### 7.1 SMOTE upsampling

**SMOTE** stands for **Synthetic Minority Oversampling Technique**. It is a method for dealing with imbalanced classification data.

The idea is simple: if one class has fewer training examples, SMOTE creates extra synthetic examples for that class. These new examples are not copied rows. They are created between existing minority-class examples and their nearest neighbours in feature space.

In this notebook, SMOTE is used only on the **training data**. The test data must stay unchanged. Otherwise, we would make the evaluation unfair.

Because SMOTE is based on distances between observations, scaling is important. Without scaling, features with large numeric ranges can dominate the distance calculation.

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**7.1 — ADVANCED TASK**

Analyze which behaviour class is predicted best and which is predicted worst.

Apply SMOTE upsampling to the training data, retrain the model, and compare the metrics before and after upsampling. Use the same untouched test set for both evaluations.

</div>

In [ ]:
# Add your code here

# Hints:
# - Compare per-class F1 from the classification report — best vs worst class
# - Build `rf_smote` pipeline: scaler → SMOTE → RandomForest (fit on resampled training data)
# - Print metrics before and after SMOTE on the **same** test set
# rf_smote = Pipeline([


Below we also plot **ROC curves** and **precision-recall curves** before and after SMOTE.

The ROC and precision-recall curves use class probabilities instead of only the final class labels. You can see this in the code below:

```python
probabilities = model.predict_proba(X)
y_score = probabilities[:, class_index]
```

Here `y_score` contains the predicted probabilities for one class.

The curves then try many possible thresholds for that one class:

```python
fpr, tpr, roc_thresholds = roc_curve(y_binary, y_score)
precision, recall, pr_thresholds = precision_recall_curve(y_binary, y_score)
```

For example, for class 2 we might ask: what happens if we predict class 2 only when its probability is above 0.50? What if we lower that threshold to 0.30? Lowering the threshold usually gives higher recall, but also more false positives. So these curves show what happens when we try many possible probability thresholds.

Why do we read these plots as **one-vs-rest**?

The model has three classes:

```text
0 = Other activity
1 = Rumination
2 = Ingestive behaviour
```

ROC and precision-recall curves are easiest to draw for a two-class question. So, for each plot, we temporarily turn the multiclass problem into a yes/no question.

For class 2, the question becomes:

```text
Is this window class 2, or not class 2?
```

That means:

```text
positive class = class 2 (Ingestive behaviour)
negative class = class 0 or class 1 (all other behaviour)
```

So a true positive is a real class-2 window that the model scores as class 2. A false positive is a class-0 or class-1 window that the model wrongly scores as class 2. A false negative is a real class-2 window that the model misses.

The same logic is repeated for class 0 and class 1. This is why the code creates `y_binary` for each class:

```python
y_binary = (y_true == cls).astype(int)
```

This line converts the true labels into 1 for the class currently being studied and 0 for all other classes.

The two plots are related, but the axes are not the same.

**ROC curve** means **Receiver Operating Characteristic curve**.

```text
ROC x-axis = false positive rate
ROC y-axis = true positive rate (recall)
```

The true positive rate is the same as recall:

```text
true positive rate = TP / (TP + FN)
```

The false positive rate is:

```text
false positive rate = FP / (FP + TN)
```

A ROC curve is useful, but it can look too optimistic when one class is rare. This happens because the false positive rate uses `TN` in the denominator. If there are many true negatives, the false positive rate can stay small even when the model still makes false positive mistakes.

A **precision-recall curve** uses recall too, but puts it on the x-axis:

```text
precision-recall x-axis = recall
precision-recall y-axis = precision
```

This plot asks a very practical question: if we lower or raise the probability threshold, how does recall change, and what happens to precision?

How do we summarise these plots?

- For the ROC curve, we use **AUC**: Area Under the ROC Curve. A larger AUC means the model separates this class from the rest better across many thresholds.
- For the precision-recall curve, we use **AP**: Average Precision. This is a summary of the precision-recall curve. A larger AP means the model keeps higher precision while recall changes.
- **Youden index** is different. It is not an area under a curve. It chooses **one threshold** on the ROC curve:

```text
Youden index = true positive rate - false positive rate
```

So AUC and AP summarise the whole curve. Youden selects one operating point on the ROC curve. That is why Youden is shown as a dot on the ROC plot and as a threshold in the table, but not as the summary number for the precision-recall plot.

We could choose a different threshold rule for the precision-recall curve, for example the threshold with the highest F1-score. But that would be another modelling choice. In this notebook we keep it simple: ROC gets AUC, precision-recall gets AP, and Youden is used later only to demonstrate threshold tuning.

In [ ]:
classes = sorted(class_mapping.keys())
model_specs = {
    'Before SMOTE': rf,
    'After SMOTE': rf_smote,
}

def collect_one_vs_rest_curves(model, X, y_true, class_labels):
    """Return ROC, PR, AUC, AP, and Youden summaries for each one-vs-rest class."""
    model_classes = model.named_steps['rf'].classes_
    probabilities = model.predict_proba(X)
    results = {}

    for cls in class_labels:
        class_index = np.where(model_classes == cls)[0][0]
        y_binary = (y_true == cls).astype(int).to_numpy()
        y_score = probabilities[:, class_index]

        fpr, tpr, roc_thresholds = roc_curve(y_binary, y_score)
        precision, recall, pr_thresholds = precision_recall_curve(y_binary, y_score)
        roc_auc = auc(fpr, tpr)
        average_precision = average_precision_score(y_binary, y_score)
        youden_index = tpr - fpr
        best_idx = int(np.argmax(youden_index))

        results[cls] = {
            'fpr': fpr,
            'tpr': tpr,
            'roc_thresholds': roc_thresholds,
            'precision': precision,
            'recall': recall,
            'pr_thresholds': pr_thresholds,
            'roc_auc': roc_auc,
            'average_precision': average_precision,
            'best_idx': best_idx,
            'best_threshold': roc_thresholds[best_idx],
            'best_fpr': fpr[best_idx],
            'best_tpr': tpr[best_idx],
            'best_youden': youden_index[best_idx],
        }

    return results

curve_results = {}
summary_rows = []

for model_name, model in model_specs.items():
    curve_results[model_name] = collect_one_vs_rest_curves(model, X_test, y_test, classes)
    for cls in classes:
        summary_rows.append({
            'model': model_name,
            'class': cls,
            'label': class_mapping.get(cls, str(cls)),
            'roc_auc': curve_results[model_name][cls]['roc_auc'],
            'average_precision': curve_results[model_name][cls]['average_precision'],
            'youden_threshold': curve_results[model_name][cls]['best_threshold'],
            'youden_fpr': curve_results[model_name][cls]['best_fpr'],
            'youden_tpr': curve_results[model_name][cls]['best_tpr'],
            'youden_index': curve_results[model_name][cls]['best_youden'],
        })

summary_df = pd.DataFrame(summary_rows).sort_values(['class', 'model'])
display(summary_df)

fig, axes = plt.subplots(len(classes), 2, figsize=(16, 4 * len(classes)), squeeze=False)

for row, cls in enumerate(classes):
    label = class_mapping.get(cls, str(cls))
    roc_ax = axes[row, 0]
    pr_ax = axes[row, 1]

    for model_name, color in [('Before SMOTE', 'tab:blue'), ('After SMOTE', 'tab:orange')]:
        data = curve_results[model_name][cls]

        roc_ax.plot(
            data['fpr'],
            data['tpr'],
            label=f'{model_name} (AUC = {data["roc_auc"]:.3f})',
            color=color,
            linewidth=2,
        )
        roc_ax.scatter(
            data['best_fpr'],
            data['best_tpr'],
            color=color,
            marker='o',
            s=45,
            zorder=3,
            label=f'{model_name} Youden threshold = {data["best_threshold"]:.3f}',
        )

        pr_ax.plot(
            data['recall'][::-1],
            data['precision'][::-1],
            label=f'{model_name} (AP = {data["average_precision"]:.3f})',
            color=color,
            linewidth=2,
        )

    roc_ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
    roc_ax.set_title(f'ROC: {label} vs rest')
    roc_ax.set_xlabel('False Positive Rate')
    roc_ax.set_ylabel('True Positive Rate')
    roc_ax.set_xlim(0, 1)
    roc_ax.set_ylim(0, 1)
    roc_ax.grid(True, alpha=0.2)
    roc_ax.legend(fontsize=9, loc='lower right')

    pr_ax.set_title(f'Precision-Recall: {label} vs rest')
    pr_ax.set_xlabel('Recall')
    pr_ax.set_ylabel('Precision')
    pr_ax.set_xlim(0, 1)
    pr_ax.set_ylim(0, 1.05)
    pr_ax.grid(True, alpha=0.2)
    pr_ax.legend(fontsize=9, loc='lower left')

plt.tight_layout()
plt.show()

<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: SMOTE effect**

**How to read this**
- SMOTE makes extra synthetic examples for the smaller classes.
- Use SMOTE only on the training data. The test data must stay untouched.
- Do not judge the model only by accuracy. Also look at macro-F1, precision, and recall for the smaller classes.
- Read ROC and precision-recall curves together. ROC shows recall against the false positive rate. The precision-recall curve shows precision against recall.

**In this notebook**
- Before SMOTE: accuracy is about 0.880 and macro-F1 is about 0.807. Class 2 has the lowest F1 score, about 0.61.
- After SMOTE: accuracy becomes a little lower, about 0.874, but macro-F1 is a little higher, about 0.810.
- The main change is for class 2. Recall improves from about 0.60 to 0.73. This means the model correctly labels more of the true ingestive-behaviour windows.
- The price is lower precision, from about 0.62 to 0.54. This means that, among the windows predicted as class 2, a larger fraction are false alarms.
- This is the trade-off the precision-recall curve helps us see: higher recall can come with lower precision.
- The ROC AUC for class 2 is still high (about 0.94), which may sound very good. But the average precision is much lower (about 0.61), which better reflects the difficulty of predicting this minority class.
- So SMOTE helps a bit with class 2 recall, but it does not solve the class imbalance problem completely.

</div>

## 8. Threshold tuning (advanced)

Until now, the reported metrics used the default Random Forest predictions from `.predict()`. In a multiclass model, this means that each test window gets the class with the highest predicted probability.

Threshold tuning changes this decision rule. Instead of always taking the class with the highest raw probability, we can use class-specific thresholds. The Youden index gives one possible threshold from the ROC curve.

For the multiclass setting here, we use those thresholds to bias the final label selection toward classes that need a lower or higher cutoff. This is mainly a teaching example: it shows how changing thresholds changes the precision-recall trade-off.

In [ ]:
# Compare default predictions with Youden-threshold-tuned predictions
def predict_with_youden_thresholds(model, X, thresholds_by_class):
    probabilities = model.predict_proba(X)
    model_classes = model.named_steps['rf'].classes_
    threshold_vector = np.array([thresholds_by_class[int(cls)] for cls in model_classes])
    adjusted_scores = probabilities - threshold_vector
    winner_index = np.argmax(adjusted_scores, axis=1)
    return model_classes[winner_index]

thresholds_before = {
    cls: curve_results['Before SMOTE'][cls]['best_threshold']
    for cls in classes
}
thresholds_after = {
    cls: curve_results['After SMOTE'][cls]['best_threshold']
    for cls in classes
}

y_pred_rf_youden = predict_with_youden_thresholds(rf, X_test, thresholds_before)
y_pred_rf_smote_youden = predict_with_youden_thresholds(rf_smote, X_test, thresholds_after)

prediction_sets = [
    ('Before SMOTE', 'Default', y_pred_rf),
    ('Before SMOTE', 'Youden tuned', y_pred_rf_youden),
    ('After SMOTE', 'Default', y_pred_rf_smote),
    ('After SMOTE', 'Youden tuned', y_pred_rf_smote_youden),
]

overall_rows = []
class_rows = []

for model_name, variant, y_pred in prediction_sets:
    report = classification_report(
        y_test,
        y_pred,
        labels=classes,
        output_dict=True,
        zero_division=0
    )
    model_variant = f'{model_name}\n{variant}'

    overall_rows.append({
        'model': model_name,
        'variant': variant,
        'model_variant': model_variant,
        'accuracy': accuracy_score(y_test, y_pred),
        'macro_f1': report['macro avg']['f1-score'],
    })

    for cls in classes:
        class_report = report[str(cls)]
        class_rows.append({
            'model': model_name,
            'variant': variant,
            'model_variant': model_variant,
            'class': cls,
            'label': class_mapping[cls],
            'precision': class_report['precision'],
            'recall': class_report['recall'],
            'f1_score': class_report['f1-score'],
            'support': int(class_report['support']),
        })

threshold_summary = pd.DataFrame(overall_rows)
class_summary = pd.DataFrame(class_rows)

print('Overall comparison')
display(
    threshold_summary[['model', 'variant', 'accuracy', 'macro_f1']].round(3)
)

print('Class-specific comparison')
display(
    class_summary[['model', 'variant', 'label', 'precision', 'recall', 'f1_score', 'support']]
    .round({'precision': 3, 'recall': 3, 'f1_score': 3})
)

# Visual comparison: overall metrics and class-specific F1 scores.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

overall_plot = threshold_summary.set_index('model_variant')[['accuracy', 'macro_f1']]
overall_plot.plot(kind='bar', ax=axes[0], ylim=(0, 1), color=['#1976D2', '#FF8F00'])
axes[0].set_title('Overall performance')
axes[0].set_ylabel('Score')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(['Accuracy', 'Macro-F1'])
axes[0].grid(axis='y', alpha=0.3)

# Print the values above the bars as well.
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.3f', padding=3, fontsize=8)

f1_matrix = class_summary.pivot(
    index='model_variant',
    columns='label',
    values='f1_score'
)
im = axes[1].imshow(f1_matrix.values, vmin=0, vmax=1, cmap='Blues')
axes[1].set_title('Class-specific F1-score')
axes[1].set_xticks(range(len(f1_matrix.columns)))
axes[1].set_xticklabels(f1_matrix.columns, rotation=30, ha='right')
axes[1].set_yticks(range(len(f1_matrix.index)))
axes[1].set_yticklabels(f1_matrix.index)

for row in range(f1_matrix.shape[0]):
    for col in range(f1_matrix.shape[1]):
        value = f1_matrix.iloc[row, col]
        axes[1].text(col, row, f'{value:.3f}', ha='center', va='center')

fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label='F1-score')
plt.tight_layout()
plt.show()


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: threshold tuning**

**How to read this**
- A model first produces probabilities. A threshold decides when a class is predicted.
- Changing the threshold can help if we care more about finding a class than about avoiding false alarms.
- The Youden index is one way to choose such a threshold from the ROC curve.
- The first table compares overall accuracy and macro-F1 for the four prediction variants.
- The second table shows class-specific precision, recall, F1-score, and support.
- The figure gives the same message visually: left = overall scores, right = class-specific F1-scores.

**In this notebook**
- Before SMOTE, Youden tuning improves macro-F1 a little, from about 0.807 to 0.815.
- Macro-F1 is the average of the F1-scores for the separate classes, with each class counted equally.
- Before SMOTE, the gain comes mainly from class 2 (`Ingestive behaviour`). Its recall improves strongly, from about 0.60 to 0.80. Its precision drops, but not enough to cancel the recall gain, so the class 2 F1-score increases from about 0.607 to 0.635.
- After SMOTE, the Youden-tuned model has the lowest macro-F1, about 0.803. This happens because the class-specific F1-scores do not improve together.
- For class 2, Youden tuning after SMOTE increases recall from about 0.731 to 0.783, but precision drops from about 0.541 to 0.502. The result is a lower class 2 F1-score, from about 0.622 to 0.612.
- Rumination also becomes slightly worse after Youden tuning: its F1-score drops from about 0.871 to 0.862. Other activity stays almost the same.
- Because macro-F1 gives equal weight to the three class-specific F1-scores, these small drops in class 2 and rumination reduce the overall macro-F1.
- The lesson is that threshold tuning can help, but it is not automatically better. A higher recall for one class is useful only if the loss in precision and the effect on the other classes are acceptable.

</div>

## 9. Cost-sensitive learning (advanced)

SMOTE changes the training data by creating synthetic minority-class examples. Cost-sensitive learning uses a different idea: keep the training data unchanged, but make mistakes on minority classes more expensive during model fitting.

For `RandomForestClassifier`, this can be done with `class_weight="balanced"`. The model automatically gives more weight to classes that occur less often in the training data. This is a useful comparison because it addresses imbalance without inventing new sensor windows.

How are these costs chosen? With `class_weight="balanced"`, scikit-learn uses a simple frequency-based rule:

```text
class weight = number of training samples / (number of classes * number of samples in that class)
```

So rare classes receive larger weights and common classes receive smaller weights. This is not a biological or economic cost. It is a pragmatic correction for unequal class frequencies. If real consequences are known, for example if missing rumination is much worse than a false rumination alarm, you could define custom class weights from those consequences. In that case, choose or tune the weights using domain knowledge and validation data, not by looking at the final test-set results.

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**9.1 — TASK**

Train a cost-sensitive Random Forest using `class_weight="balanced"`.

Compare the main imbalance strategies in one table and one plot:

- default Random Forest;
- Random Forest with SMOTE;
- threshold-tuned Random Forest;
- cost-sensitive Random Forest.

Focus on macro-F1 and class-specific F1. Accuracy alone can still hide poor performance on smaller classes.

</div>

In [ ]:
# Add your code here

# Hints:
# - Build `rf_weighted` with `class_weight="balanced"` inside `RandomForestClassifier`.
# - Fit it on the original `X_train`, `y_train`; do not apply SMOTE in this model.
# - Store predictions in `y_pred_rf_weighted`.
# - Compare all strategies using the same unchanged `X_test`, `y_test`.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: comparing imbalance strategies**

**How to read this**

The table and plots compare four ways of handling the same imbalanced classification problem. All models are evaluated on the same unchanged test set, so differences in the scores come from the modelling strategy, not from changing the test data.

- **Macro-F1** gives each behaviour class equal weight. This is usually the most useful single summary here because the smaller classes matter as much as the majority class.
- **Weighted-F1** gives larger classes more influence. If weighted-F1 is much higher than macro-F1, the model is probably doing better on common behaviours than on rare behaviours.
- **Class-specific F1** shows which behaviours improved or became worse. This is important because an imbalance method can improve one minority class while hurting another class.

**What to look for**

Do not only choose the model with the highest accuracy. First check whether macro-F1 improves. Then inspect the class-specific F1 plot. A good imbalance strategy should improve the weaker behaviour classes without causing a large drop for the other behaviours.

**In this notebook**

- **SMOTE** changes the training data by creating synthetic minority examples.
- **Threshold tuning** changes the decision rule after the model has produced probabilities.
- **Cost-sensitive learning** keeps the data unchanged, but changes how strongly the model penalizes mistakes on each class.

Here the cost-sensitive model uses automatic frequency-based weights from the training set. These weights are a reasonable default when no real application-specific cost matrix is available.

If the cost-sensitive model performs similarly to SMOTE, it may be preferable because it is simpler and does not create artificial sensor windows. If SMOTE performs better, check whether the improvement is broad across classes or mainly driven by one behaviour.

</div>

## 10. Confusion matrix evaluation

### 10.1 Confusion matrices

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**10.1 — TASK**

Plot the confusion matrix for the **default** Random Forest predictions before and after SMOTE.

Use:

- `y_pred_rf` for the model without SMOTE;
- `y_pred_rf_smote` for the model with SMOTE.

Do **not** use the Youden-tuned predictions here. Those were included in section 8 to demonstrate threshold tuning. For the main confusion-matrix comparison, we keep the simpler default models.

Use a heatmap for better visualization.

</div>

#### Without SMOTE

In [ ]:
# Add your code here

# Hints:
# - Use `y_pred_rf` for the default model without SMOTE.
# - Do not use `y_pred_rf_youden` for the main confusion matrix.
# - Plot with `ConfusionMatrixDisplay.from_predictions(...)` or `seaborn.heatmap`.
# - Use behaviour labels from `class_mapping`.
# - Normalise optionally by row to see recall per true class.


#### With SMOTE

In [ ]:
# Add your code here

# Hints:
# - Use `y_pred_rf_smote` for the default SMOTE model.
# - Do not use `y_pred_rf_smote_youden` for the main confusion matrix.
# - Use the same label order and colour scale as the first heatmap for easy comparison.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation — confusion matrices**

**General guidelines**
- Diagonal cells are correct predictions; off-diagonal cells show confusion between behaviours.
- Compare raw counts and normalised matrices when classes are imbalanced.
- Look for systematic confusions (e.g. rumination vs other activity).

**This notebook's output**
- Side-by-side heatmaps compare the default `rf` predictions with the default `rf_smote` predictions on the same 7200 test windows.
- With SMOTE, more windows are predicted as class 2 (ingestive), matching the recall gain seen in the classification report.
- Majority class 0 still accounts for most correct predictions.

</div>

## 11. Feature importance (advanced)

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**11.1 — ADVANCED TASK**

Analyze and plot feature importances for the SMOTE random forest model.

</div>

In [ ]:
# Add your code here

# Hints:
# - Access importances via `rf_smote.named_steps['rf'].feature_importances_`.
# - Pair with `X_train.columns` and sort descending.
# - Make a bar plot of the top features.
# - Compare the important features with the feature-vs-class plots in section 4.1.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: feature importances**

**How to read this**
- Random Forest importances show which features the model used most often for splitting.
- They are useful for interpretation, but they are not proof of a causal effect.
- Always compare the result with what you know about the sensors and the animal behaviour.

**In this notebook**
- The three most important features are `odba` (about 0.27), `range_pressure` (about 0.26), and `svm_std` (about 0.20).
- `odba` means **Overall Dynamic Body Acceleration**. It summarizes how much the animal is moving after removing the slower/static part of the acceleration signal. A high `odba` value means stronger body movement within the window.
- `svm_std` is the standard deviation of the **Signal Vector Magnitude** within a window. The Signal Vector Magnitude combines movement in the x, y, and z directions into one movement-strength value. The standard deviation tells us how variable that movement strength is during the window.
- In simple terms, `odba` captures the amount of movement, while `svm_std` captures how much the movement changes within the window.
- `range_pressure` measures the difference between the highest and lowest pressure value in the window. This can reflect changes in jaw or noseband pressure during different behaviours.
- This fits with the feature-vs-class plots from section 4.1. Movement intensity, movement variability, and pressure variation help distinguish behaviours.
- Mean pressure is less important than changes in movement and pressure. This suggests that the model learns more from dynamic patterns than from the average pressure level alone.

</div>

## 12. Model visualisation and interpretation

<div style="background:#000000;border-left:5px solid #1976D2;padding:14px 18px;margin:14px 0;border-radius:0 4px 4px 0;">

**12.1 — TASK**

Make a line plot of the **reference-label** and predicted minutes spent ruminating per hour for the test set. Use the best model from the comparison table in section 9.

Here, "reference-label" means the rumination labels already present in the dataset (`y_test`). These are the labels we compare the model against.

The x-axis should represent the time (in hours) and the y-axis should represent the minutes spent ruminating. Include a legend to differentiate between reference-label and predicted values.

</div>

In [ ]:
# Add your code here

# Hints:
# - Aggregate reference-label rumination minutes per hour on the test set (class 1 in `y_test`).
# - Select the best model name from `model_comparison_df` using macro-F1.
# - Get its predictions from `comparison_predictions` for the predicted series.
# - Line plot with legend: reference-label vs predicted.


<div style="background:#000000;border-left:5px solid #FF8F00;padding:12px 16px;margin:10px 0;border-radius:0 4px 4px 0;">

**Interpretation: rumination time plot**

**How to read this**
- Window-level predictions are useful, but farmers and researchers often think in larger time units.
- Minutes of rumination per hour is easier to understand than thousands of individual window labels.
- A time plot can show whether the model follows the daily pattern or misses whole rumination periods.

**In this notebook**
- The orange line is calculated from the reference labels in `y_test`. These are the labels supplied with the dataset, not necessarily direct human observations.
- The blue line is calculated from the model predictions.
- If the two lines follow each other, the model captures the main rumination pattern according to the available reference labels.
- If the predicted line is often too low or too high, the model has a systematic bias for rumination relative to those reference labels.

</div>